# Lab 3 

## Activity 1

In [1]:
import re

customer_message = """
Hello, I am writing to inquire about my recent orders.
My first order number is ORD-58392 and the second one is ORD-99201.
You can reach me at john.doe@email.com or jane_doe123@work.org.
My phone numbers are 555-123-4567 and (800) 555-9999.
I paid $45.99 for 2 items and $120 for 3 packages.
"""

emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", customer_message)
orders = re.findall(r"ORD-\d+", customer_message)
phones = re.findall(r"(?:\(\d{3}\)|\d{3})[-.\s]*\d{3}[-.\s]*\d{4}", customer_message)

print("Emails:", emails)
print("Order numbers:", orders)
print("Phone numbers:", phones)

Emails: ['john.doe@email.com', 'jane_doe123@work.org']
Order numbers: ['ORD-58392', 'ORD-99201']
Phone numbers: ['555-123-4567', '(800) 555-9999']


## Activity 2

In [2]:
import spacy

nlp = spacy.blank("en")
doc = nlp(customer_message)

currency_amounts = [
    token.text + doc[i + 1].text
    for i, token in enumerate(doc[:-1])
    if token.is_currency and doc[i + 1].like_num
]

quantities = [
    token.text + " " + doc[i + 1].text
    for i, token in enumerate(doc[:-1])
    if token.like_num and doc[i + 1].lower_ in {"items", "packages"}
]

regex_amounts = re.findall(r"\$\d+(?:,\d{3})*(?:\.\d+)?", customer_message)
regex_quantities = re.findall(r"\b\d+\s+(?:items|packages)\b", customer_message)

print("spaCy currency amounts:", currency_amounts)
print("Regex currency amounts:", regex_amounts)
print("spaCy quantities:", quantities)
print("Regex quantities:", regex_quantities)
print("Currency matches:", currency_amounts == regex_amounts)
print("Quantity matches:", quantities == regex_quantities)

spaCy currency amounts: ['$45.99', '$120']
Regex currency amounts: ['$45.99', '$120']
spaCy quantities: ['2 items', '3 packages']
Regex quantities: ['2 items', '3 packages']
Currency matches: True
Quantity matches: True


## Activity 3

In [3]:
from nltk.tokenize import word_tokenize
from difflib import SequenceMatcher

sample_document = "I'm learning NLP, aren't you? Let's test contractions, emails like user@example.com, and $20.00 prices!"

nltk_tokens = word_tokenize(sample_document, preserve_line=True)
spacy_tokens = [token.text for token in nlp(sample_document)]

print("NLTK tokens:", nltk_tokens)
print("spaCy tokens:", spacy_tokens)
print("\nDifferences:")

found_difference = False
for kind, i1, i2, j1, j2 in SequenceMatcher(None, nltk_tokens, spacy_tokens).get_opcodes():
    if kind != "equal":
        print("NLTK:", nltk_tokens[i1:i2], "| spaCy:", spacy_tokens[j1:j2])
        found_difference = True

if not found_difference:
    print("No differences")

NLTK tokens: ['I', "'m", 'learning', 'NLP', ',', 'are', "n't", 'you', '?', 'Let', "'s", 'test', 'contractions', ',', 'emails', 'like', 'user', '@', 'example.com', ',', 'and', '$', '20.00', 'prices', '!']
spaCy tokens: ['I', "'m", 'learning', 'NLP', ',', 'are', "n't", 'you', '?', 'Let', "'s", 'test', 'contractions', ',', 'emails', 'like', 'user@example.com', ',', 'and', '$', '20.00', 'prices', '!']

Differences:
NLTK: ['user', '@', 'example.com'] | spaCy: ['user@example.com']


## Activity 4

In [4]:
from nltk.tokenize import PunktSentenceTokenizer

paragraph = (
    "Natural language processing is useful. "
    "Dr. Khan studies language models. "
    "He works with NLTK and spaCy. "
    "Don't confuse a word with a sentence! "
    "Can software detect every boundary? "
    "We will compare the results today."
)

nltk_sentences = PunktSentenceTokenizer(paragraph).tokenize(paragraph)
sentence_nlp = spacy.blank("en")
sentence_nlp.add_pipe("sentencizer")
spacy_sentences = [sentence.text for sentence in sentence_nlp(paragraph).sents]

print("NLTK sentences:")
for i, sentence in enumerate(nltk_sentences, 1):
    print(i, sentence)

print("\nspaCy sentences:")
for i, sentence in enumerate(spacy_sentences, 1):
    print(i, sentence)

print("\nMismatch:", nltk_sentences != spacy_sentences)
if nltk_sentences != spacy_sentences:
    for i in range(max(len(nltk_sentences), len(spacy_sentences))):
        first = nltk_sentences[i] if i < len(nltk_sentences) else "[missing]"
        second = spacy_sentences[i] if i < len(spacy_sentences) else "[missing]"
        if first != second:
            print(f"Sentence {i + 1}:")
            print("NLTK:", first)
            print("spaCy:", second)

NLTK sentences:
1 Natural language processing is useful.
2 Dr. Khan studies language models.
3 He works with NLTK and spaCy.
4 Don't confuse a word with a sentence!
5 Can software detect every boundary?
6 We will compare the results today.

spaCy sentences:
1 Natural language processing is useful.
2 Dr. Khan studies language models.
3 He works with NLTK and spaCy.
4 Don't confuse a word with a sentence!
5 Can software detect every boundary?
6 We will compare the results today.

Mismatch: False
